# 实验1：ViT-B/16 在 CIFAR-10 上（预训练 vs 不预训练）
这个 notebook 版本避免 argparse 与 ipykernel 的 `-f xxx.json` 参数冲突。

特点：
- `download=False`：假设你已准备好 CIFAR-10
- 逐块运行，每块末尾都会 `print('OK: ...')` 便于确认
- 输出：`logs/` 下保存 best/last/summary

使用方法：从上到下依次运行。需要切换预训练/不预训练时，改动配置块里的 `PRETRAINED`。

In [1]:

# ====== 0. 配置区（你主要改这里） ======
DATA_ROOT = "./data"      # 这里指向包含 cifar-10-batches-py 的目录
PRETRAINED = True         # True=加载 ImageNet 预训练权重；False=从零开始
EPOCHS = 10               # 先小一点验证流程
BATCH_SIZE = 128
LR = 3e-4
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
VAL_RATIO = 0.1
NUM_WORKERS = 4
SEED = 42
AMP = True                # 仅 CUDA 有效；CPU 自动关闭

print("OK: 配置已加载", {"PRETRAINED": PRETRAINED, "EPOCHS": EPOCHS, "DATA_ROOT": DATA_ROOT})


OK: 配置已加载 {'PRETRAINED': True, 'EPOCHS': 10, 'DATA_ROOT': './data'}


In [2]:

# ====== 1. 导入与环境检查 ======
import os, time, json, random
from dataclasses import dataclass
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision.datasets import CIFAR10
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = bool(AMP) and (DEVICE.type == "cuda")
SCALER = torch.cuda.amp.GradScaler() if AMP else None

print("OK: 环境就绪", {"torch": torch.__version__, "torchvision": torchvision.__version__, "device": str(DEVICE), "amp": AMP})


OK: 环境就绪 {'torch': '2.9.1+cu130', 'torchvision': '0.24.1+cu130', 'device': 'cuda', 'amp': True}


/tmp/ipykernel_11504/3183475753.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  SCALER = torch.cuda.amp.GradScaler() if AMP else None


In [3]:

# ====== 2. 复现 & 日志工具 ======
def set_seed(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class SimpleLogger:
    def __init__(self, log_path: str):
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        self.f = open(log_path, "a", encoding="utf-8")
    def log(self, msg: str):
        t = time.strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{t}] {msg}"
        print(line, flush=True)
        self.f.write(line + "\n")
        self.f.flush()
    def close(self):
        self.f.close()

@torch.no_grad()
def accuracy_top1(logits: torch.Tensor, targets: torch.Tensor) -> float:
    return (logits.argmax(dim=1) == targets).float().mean().item()

def save_checkpoint(path: str, model: nn.Module, optimizer: optim.Optimizer, scaler, epoch: int, best_acc: float, config: dict):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": None if scaler is None else scaler.state_dict(),
        "epoch": epoch,
        "best_acc": best_acc,
        "config": config,
    }, path)

def load_checkpoint(path: str, model: nn.Module, optimizer: Optional[optim.Optimizer] = None, scaler=None, map_location="cpu"):
    ckpt = torch.load(path, map_location=map_location)
    model.load_state_dict(ckpt["model"])
    if optimizer is not None and ckpt.get("optimizer") is not None:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scaler is not None and ckpt.get("scaler") is not None:
        scaler.load_state_dict(ckpt["scaler"])
    return ckpt

set_seed(SEED)
print("OK: 工具函数已定义 & 随机种子已固定", {"SEED": SEED})


OK: 工具函数已定义 & 随机种子已固定 {'SEED': 42}


In [4]:

# ====== 3. 数据集与 DataLoader（download=False） ======
def build_transforms(pretrained: bool):
    # 统一 resize 到 224（vit_b_16 默认输入）
    if pretrained:
        # 不再依赖 weights.transforms() 的内部结构（不同 torchvision 版本不一致）
        imagenet_mean = (0.485, 0.456, 0.406)
        imagenet_std  = (0.229, 0.224, 0.225)

        train_tf = transforms.Compose([
            transforms.Resize(232),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.AutoAugment(policy=transforms.AutoAugmentPolicy.IMAGENET),
            transforms.ToTensor(),
            transforms.Normalize(imagenet_mean, imagenet_std),
        ])
        val_tf = transforms.Compose([
            transforms.Resize(232),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(imagenet_mean, imagenet_std),
        ])
        return train_tf, val_tf

    # ===== 不预训练分支保持你原来的就行 =====
    imagenet_mean = (0.485, 0.456, 0.406)
    imagenet_std = (0.229, 0.224, 0.225)
    train_tf = transforms.Compose([
        transforms.Resize(232),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.AutoAugment(policy=transforms.AutoAugmentPolicy.CIFAR10),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(232),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])
    return train_tf, val_tf

def build_dataloaders(data_root: str, batch_size: int, num_workers: int, pretrained: bool, val_ratio: float, seed: int):
    train_tf, val_tf = build_transforms(pretrained=pretrained)

    full_train = CIFAR10(root=data_root, train=True, transform=train_tf, download=False)
    full_val = CIFAR10(root=data_root, train=True, transform=val_tf, download=False)

    n = len(full_train)
    n_val = int(n * val_ratio)
    n_train = n - n_val

    g = torch.Generator().manual_seed(seed)
    train_idx, val_idx = random_split(range(n), [n_train, n_val], generator=g)

    train_set = torch.utils.data.Subset(full_train, train_idx.indices)
    val_set = torch.utils.data.Subset(full_val, val_idx.indices)
    test_set = CIFAR10(root=data_root, train=False, transform=val_tf, download=False)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader

try:
    train_loader, val_loader, test_loader = build_dataloaders(
        DATA_ROOT, BATCH_SIZE, NUM_WORKERS, PRETRAINED, VAL_RATIO, SEED
    )
    print("OK: DataLoader 构建成功", {"train_batches": len(train_loader), "val_batches": len(val_loader), "test_batches": len(test_loader)})
except Exception as e:
    print("❌ 数据加载失败：请检查 DATA_ROOT 是否正确，并确认 download=False 下本地数据已存在。")
    raise


OK: DataLoader 构建成功 {'train_batches': 352, 'val_batches': 40, 'test_batches': 79}


In [5]:

# ====== 4. 构建 ViT-B/16 模型（分类头改为 10 类） ======
def build_model(pretrained: bool, num_classes: int = 10) -> nn.Module:
    if pretrained:
        weights = ViT_B_16_Weights.IMAGENET1K_V1
        model = vit_b_16(weights=weights)
    else:
        model = vit_b_16(weights=None)

    if hasattr(model, "heads"):
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, num_classes)
    else:
        in_features = model.head.in_features
        model.head = nn.Linear(in_features, num_classes)
    return model

model = build_model(PRETRAINED, num_classes=10).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("OK: 模型/损失/优化器/调度器 已就绪", {"pretrained": PRETRAINED})


OK: 模型/损失/优化器/调度器 已就绪 {'pretrained': True}


In [6]:

# ====== 5. 训练/验证函数 ======
@dataclass
class Stats:
    loss: float
    acc: float

def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: optim.Optimizer, criterion: nn.Module,
                    device: torch.device, scaler, amp: bool, logger: SimpleLogger, log_interval: int = 50) -> Stats:
    model.train()
    total_loss, total_acc, n = 0.0, 0.0, 0
    for i, (x, y) in enumerate(loader):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        if amp and scaler is not None:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        acc = accuracy_top1(logits.detach(), y)
        total_loss += loss.item()
        total_acc += acc
        n += 1

        if (i + 1) % log_interval == 0:
            logger.log(f"  [train] iter {i+1}/{len(loader)} | loss={total_loss/n:.4f} acc={total_acc/n*100:.2f}%")
    return Stats(total_loss/max(1,n), total_acc/max(1,n))

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> Stats:
    model.eval()
    total_loss, total_acc, n = 0.0, 0.0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)
        acc = accuracy_top1(logits, y)
        total_loss += loss.item()
        total_acc += acc
        n += 1
    return Stats(total_loss/max(1,n), total_acc/max(1,n))

print("OK: 训练/验证函数已定义")


OK: 训练/验证函数已定义


In [7]:

# ====== 6. 正式训练（保存 best / last / summary.json） ======
exp_name = f"exp1_vit_nb_pretrained{int(PRETRAINED)}_e{EPOCHS}_bs{BATCH_SIZE}_lr{LR}"
out_dir = os.path.join("logs", exp_name)
os.makedirs(out_dir, exist_ok=True)
logger = SimpleLogger(os.path.join(out_dir, "train.log"))

config = {
    "DATA_ROOT": DATA_ROOT, "PRETRAINED": PRETRAINED, "EPOCHS": EPOCHS,
    "BATCH_SIZE": BATCH_SIZE, "LR": LR, "WEIGHT_DECAY": WEIGHT_DECAY,
    "LABEL_SMOOTHING": LABEL_SMOOTHING, "VAL_RATIO": VAL_RATIO, "NUM_WORKERS": NUM_WORKERS,
    "SEED": SEED, "AMP": AMP, "DEVICE": str(DEVICE)
}
logger.log("==== Notebook 实验1：ViT-B/16 on CIFAR-10 ====")
logger.log(f"config = {config}")

best_val_acc = 0.0

for epoch in range(EPOCHS):
    logger.log(f"\n[epoch {epoch+1}/{EPOCHS}] lr={optimizer.param_groups[0]['lr']:.6f}")
    tr = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, SCALER, AMP, logger, log_interval=50)
    va = evaluate(model, val_loader, criterion, DEVICE)
    logger.log(f"[epoch {epoch+1}] train: loss={tr.loss:.4f}, acc={tr.acc*100:.2f}%")
    logger.log(f"[epoch {epoch+1}]   val: loss={va.loss:.4f}, acc={va.acc*100:.2f}%")

    save_checkpoint(os.path.join(out_dir, "last.pt"), model, optimizer, SCALER, epoch, best_val_acc, config)
    if va.acc > best_val_acc:
        best_val_acc = va.acc
        save_checkpoint(os.path.join(out_dir, "best.pt"), model, optimizer, SCALER, epoch, best_val_acc, config)
        logger.log(f"⭐ New best val acc: {best_val_acc*100:.2f}% -> saved best.pt")

    scheduler.step()

logger.log("\n==== 测试集评估（best.pt）====")
best_path = os.path.join(out_dir, "best.pt")
if os.path.exists(best_path):
    _ = load_checkpoint(best_path, model, optimizer=None, scaler=None, map_location=DEVICE)

te = evaluate(model, test_loader, criterion, DEVICE)
logger.log(f"[test] loss={te.loss:.4f}, acc={te.acc*100:.2f}%")

summary = {"exp_name": exp_name, "best_val_acc": best_val_acc, "test_acc": te.acc, **config}
with open(os.path.join(out_dir, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

logger.log(f"✅ 完成。输出目录：{out_dir}")
logger.close()

print("OK: 训练完成", {"best_val_acc": best_val_acc, "test_acc": te.acc, "out_dir": out_dir})


[2026-03-05 10:42:06] ==== Notebook 实验1：ViT-B/16 on CIFAR-10 ====
[2026-03-05 10:42:06] config = {'DATA_ROOT': './data', 'PRETRAINED': True, 'EPOCHS': 10, 'BATCH_SIZE': 128, 'LR': 0.0003, 'WEIGHT_DECAY': 0.05, 'LABEL_SMOOTHING': 0.1, 'VAL_RATIO': 0.1, 'NUM_WORKERS': 4, 'SEED': 42, 'AMP': True, 'DEVICE': 'cuda'}
[2026-03-05 10:42:06] 
[epoch 1/10] lr=0.000300
[2026-03-05 10:42:25]   [train] iter 50/352 | loss=2.2882 acc=13.83%
[2026-03-05 10:42:42]   [train] iter 100/352 | loss=2.2055 acc=18.27%
[2026-03-05 10:42:59]   [train] iter 150/352 | loss=2.1229 acc=22.69%
[2026-03-05 10:43:17]   [train] iter 200/352 | loss=2.0343 acc=27.39%
[2026-03-05 10:43:34]   [train] iter 250/352 | loss=1.9290 acc=32.90%
[2026-03-05 10:43:51]   [train] iter 300/352 | loss=1.8184 acc=38.57%
[2026-03-05 10:44:09]   [train] iter 350/352 | loss=1.7148 acc=43.70%
[2026-03-05 10:44:21] [epoch 1] train: loss=1.7114, acc=43.87%
[2026-03-05 10:44:21] [epoch 1]   val: loss=0.9076, acc=82.93%
[2026-03-05 10:44:35] ⭐ 